In [2]:
# Fine-tune a 3B model (like Qwen-2.5-Coder) to not just write code, but to explain and fix its own bugs based on error logs.

# Deep Learning Tech: * SFT: Train the model on the (Buggy Code + Error Log) -> Fixed Code pattern.

# DPO (Preference Training): Give the model two versions of a fix—one that is "hacky" and one that follows best practices. Use DPO to force it to prefer the "Clean Code" version.

In [3]:
%%capture
!pip install unsloth

In [ ]:
HF_token = 'hf_.....'

In [5]:
## reading the dataset 
import pandas as pd
# Paste the copied file path inside the quotes
file_path = "/kaggle/input/code-finetuning-dataset/code_fixex_finetuning_dataset.csv" 
df = pd.read_csv(file_path)

# Display the first few rows of the DataFrame
print(f"The columns in the Dataframe are ==>{df.columns}<===")
print(f"The first 2 row in dataset are ===>{df.head(2)}<===")

The columns in the Dataframe are ==>Index(['bug_type', 'buggy_code', 'error_log', 'fixed_code', 'explanation'], dtype='object')<===
The first 2 row in dataset are ===>        bug_type                               buggy_code  \
0   Syntax Error  def add_numbers(a, b)\n    return a + b   
1  Runtime Error   numbers = [1, 2, 3]\nprint(numbers[3])   

                             error_log  \
0            SyntaxError: expected ':'   
1  IndexError: list index out of range   

                                 fixed_code  \
0  def add_numbers(a, b):\n    return a + b   
1    numbers = [1, 2, 3]\nprint(numbers[2])   

                                         explanation  
0  The function definition is missing a colon at ...  
1  The list has only three elements, so accessing...  <===


In [6]:
df.head(2)   ## looks better here

,bug_type,buggy_code,error_log,fixed_code,explanation
0,Syntax Error,"def add_numbers(a, b)\n return a + b",SyntaxError: expected ':',"def add_numbers(a, b):\n return a + b",The function definition is missing a colon at ...
1,Runtime Error,"numbers = [1, 2, 3]\nprint(numbers[3])",IndexError: list index out of range,"numbers = [1, 2, 3]\nprint(numbers[2])","The list has only three elements, so accessing..."


## first comes the dataset conversion

In [7]:
## first convert the padas dataframe to dict for consumption in to the dataset method required for SFT

data_dict = df.to_dict(orient='records')

# dataset = Dataset.from_dict

In [8]:
## to check the model is using any chat template
from transformers import AutoTokenizer
model_id = 'Qwen/Qwen2.5-Coder-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

print(tokenizer.chat_template)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [9]:
SYSTEM_PROMPT = (
    "You are an expert software engineer. "
    "You analyze buggy code, understand error logs, explain the root cause clearly, "
    "and then provide a clean, best-practice fix."
)

def preprocess_function(example):
    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Bug Type: {example['bug_type']}\n\n"
                    f"Buggy Code:\n```python\n{example['buggy_code']}\n```\n\n"
                    f"Error Log:\n{example['error_log']}"
                )
            },
            {
                "role": "assistant",
                "content": (
                    f"Explanation:\n{example['explanation']}\n\n"
                    f"Fixed Code:\n```python\n{example['fixed_code']}\n```"
                )
            }
        ]
    }
    

In [10]:
# converting to dict data_dict to dataset

from datasets import Dataset
# data_dict from csv is like this [{},{},..] so for loading it we cant use from_dict instead use from_list

dataset = Dataset.from_list(data_dict)
dataset = dataset.map(preprocess_function, remove_columns = ["bug_type","buggy_code","error_log","fixed_code","explanation"])

Map:   0%|          | 0/204 [00:00<?, ? examples/s]

In [11]:
### till now we have processed the data already lets load the model and tokeinzer
from unsloth import FastLanguageModel
import torch

model,tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    load_in_4bit = True,
    token = HF_token,
    load_in_8bit = False,
    full_finetuning = False,
    use_gradient_checkpointing = 'unsloth'
)

/tmp/ipykernel_55/3584429076.py:2: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-27 12:11:39.604469: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766837499.797588      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766837499.852499      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766837500.314717      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766837500.314752      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766837500.314755      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Qwen2 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
## always after loading the model checkout its attention layer
## to understand the target modules
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen

## 1. from_pretrained - loads the base model
## 2. get_peft_model - modifies that model
    - Injects LoRA adapters

    - Makes only small matrices trainable
    
    - Keeps VRAM usage low
    
    - Enables fine-tuning

In [13]:
model = FastLanguageModel.get_peft_model(
    model = model,
    r = 32, ## rank of a matrix,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # it support rank stabilized LoRA
    loftq_config = None,
)

Unsloth 2025.12.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [14]:
#apply the chat template to the conversations

chat_texts = tokenizer.apply_chat_template(
    list(dataset["messages"]),
    tokenize=False,
)

In [15]:
## creating a training dtaset

train_dataset = Dataset.from_dict({
    "text": chat_texts
})

## Fine Tune the Model-

In [16]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = None,

    args = SFTConfig(
        dataset_text_field = "text",
        packing = True,  # IMPORTANT for speed + memory

        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,

        warmup_steps = 5,
        max_steps = 30,  # small test run

        learning_rate = 2e-4,
        logging_steps = 1,

        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",

        fp16 = True,     # T4-safe
        bf16 = False,

        save_strategy = "steps",
        save_steps = 30,

        seed = 3407,
        report_to = "none",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/204 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/204 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [17]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
2.971 GB of memory reserved.


In [18]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16 | Num Epochs = 15 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.000400
2,3.226000
3,2.969300
4,2.901000
5,2.593200
6,2.197200
7,1.782000
8,1.441100
9,1.191100
10,0.939400


## Show final memory and time stats



In [19]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

563.5573 seconds used for training.
9.39 minutes used for training.
Peak reserved memory = 5.893 GB.
Peak reserved memory for training = 2.922 GB.
Peak reserved memory % of max memory = 39.977 %.
Peak reserved memory for training % of max memory = 19.822 %.


In [43]:
## SFT model trained-

messages = [
    {"role" : "user", "content" : """I have this Python class, but it throws an error when I try to create an object and call the method:

```python
class Calculator:
    def __init__(self, numbers):
        self.numbers = numbers
    
    def sum_all(sel):
        total = 0
        for n in self.numbers:
            total += n
        return total

calc = Calculator(5, 10, 15)
print(calc.sum_all())"""}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Explanation:

In Python, when defining methods, you need to use `def` instead of `sel`. Also, when calling the method, you need to pass the correct argument name `sel` as `self`.<|im_end|>


In [21]:
dataset[0]

{'messages': [{'content': 'You are an expert software engineer. You analyze buggy code, understand error logs, explain the root cause clearly, and then provide a clean, best-practice fix.',
   'role': 'system'},
  {'content': "Bug Type: Syntax Error\n\nBuggy Code:\n```python\ndef add_numbers(a, b)\n    return a + b\n```\n\nError Log:\nSyntaxError: expected ':'",
   'role': 'user'},
  {'content': 'Explanation:\nThe function definition is missing a colon at the end of the parameter list, which is required by Python syntax.\n\nFixed Code:\n```python\ndef add_numbers(a, b):\n    return a + b\n```',
   'role': 'assistant'}]}

In [31]:
def generate_dpo_pair(example,
                      max_tokens_chosen=128,
                      max_tokens_rejected=32):

    # Convert prompt messages → text
    prompt_text = tokenizer.apply_chat_template(
        example["prompt_messages"],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
    prompt_len = inputs["input_ids"].shape[-1]

    # -------- chosen --------
    chosen_ids = model.generate(
        **inputs,
        max_new_tokens=max_tokens_chosen,
        do_sample=False,
    )
    chosen = tokenizer.decode(
        chosen_ids[0][prompt_len:], skip_special_tokens=True
    ).strip()

    # -------- rejected --------
    rejected_ids = model.generate(
        **inputs,
        max_new_tokens=max_tokens_rejected,
        do_sample=False,
    )
    rejected = tokenizer.decode(
        rejected_ids[0][prompt_len:], skip_special_tokens=True
    ).strip()

    return {
        "prompt": prompt_text,   # STRING
        "chosen": chosen,        # STRING
        "rejected": rejected,    # STRING
    }

dataset_dpo = dataset.map(generate_dpo_pair, batched=False)
print(dataset_dpo[0])

Map:   0%|          | 0/204 [00:00<?, ? examples/s]

{'messages': [{'content': 'You are an expert software engineer. You analyze buggy code, understand error logs, explain the root cause clearly, and then provide a clean, best-practice fix.', 'role': 'system'}, {'content': "Bug Type: Syntax Error\n\nBuggy Code:\n```python\ndef add_numbers(a, b)\n    return a + b\n```\n\nError Log:\nSyntaxError: expected ':'", 'role': 'user'}, {'content': 'Explanation:\nThe function definition is missing a colon at the end of the parameter list, which is required by Python syntax.\n\nFixed Code:\n```python\ndef add_numbers(a, b):\n    return a + b\n```', 'role': 'assistant'}], 'prompt_messages': [{'content': 'You are an expert software engineer. You analyze buggy code, understand error logs, explain the root cause clearly, and then provide a clean, best-practice fix.', 'role': 'system'}, {'content': "Bug Type: Syntax Error\n\nBuggy Code:\n```python\ndef add_numbers(a, b)\n    return a + b\n```\n\nError Log:\nSyntaxError: expected ':'", 'role': 'user'}], '

In [32]:
print(dataset_dpo[3])

{'messages': [{'content': 'You are an expert software engineer. You analyze buggy code, understand error logs, explain the root cause clearly, and then provide a clean, best-practice fix.', 'role': 'system'}, {'content': 'Bug Type: Bad Practice\n\nBuggy Code:\n```python\ndef add_item(item, items=[]):\n    items.append(item)\n    return items\n```\n\nError Log:\nUnexpected accumulation of items across function calls', 'role': 'user'}, {'content': 'Explanation:\nUsing a mutable default argument causes the same list to be reused across calls, leading to unintended behavior.\n\nFixed Code:\n```python\ndef add_item(item, items=None):\n    if items is None:\n        items = []\n    items.append(item)\n    return items\n```', 'role': 'assistant'}], 'prompt_messages': [{'content': 'You are an expert software engineer. You analyze buggy code, understand error logs, explain the root cause clearly, and then provide a clean, best-practice fix.', 'role': 'system'}, {'content': 'Bug Type: Bad Practi

In [33]:
dataset_dpo[0].keys()

dict_keys(['messages', 'prompt_messages', 'prompt', 'chosen', 'rejected'])

In [29]:
def add_prompt_messages(example):
    return {
        "prompt_messages": [
            m for m in example["messages"]
            if m["role"] != "assistant"
        ]
    }

dataset = dataset.map(add_prompt_messages)

Map:   0%|          | 0/204 [00:00<?, ? examples/s]

In [34]:
dataset_dpo = dataset_dpo.remove_columns(
    [c for c in dataset_dpo.column_names if c not in ["prompt", "chosen", "rejected"]]
)

In [46]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_args = DPOConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    num_train_epochs=3,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    seed=42,
    output_dir="outputs",
)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = dpo_args,
    beta = 0.1,
    train_dataset = dataset_dpo,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

dpo_trainer.train()


Extracting prompt in train dataset (num_proc=8):   0%|          | 0/204 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=8):   0%|          | 0/204 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=8):   0%|          | 0/204 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 204 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.689100,5.086769,5.061314,0.468750,0.025455,-9.472065,-25.765179,-3.645451,-3.695853,0,0,0
2,0.620600,5.387759,5.224525,0.750000,0.163234,-8.741070,-30.220072,-3.670346,-3.719318,No Log,No Log,No Log
3,0.662800,4.865932,4.777018,0.625000,0.088915,-9.350958,-28.363274,-3.661460,-3.721819,No Log,No Log,No Log
4,0.568400,5.259257,4.976748,0.750000,0.282508,-8.820779,-30.035942,-3.827994,-3.900391,No Log,No Log,No Log
5,0.518500,5.508901,5.094132,0.781250,0.414769,-8.256271,-29.719004,-4.133292,-4.198936,No Log,No Log,No Log
6,0.451600,5.253547,4.646256,0.781250,0.607291,-7.543856,-31.252956,-4.322046,-4.339280,No Log,No Log,No Log
7,0.414700,5.106087,4.392442,0.833333,0.713646,-7.407997,-33.950111,-4.605866,-4.629047,No Log,No Log,No Log
8,0.355900,5.348464,4.383440,0.812500,0.965023,-6.667825,-33.245407,-5.167974,-5.213895,No Log,No Log,No Log
9,0.349000,5.340741,4.309530,0.750000,1.031211,-7.503562,-36.203022,-5.427945,-5.510690,No Log,No Log,No Log
10,0.226900,5.094255,3.577286,0.906250,1.516970,-7.864432,-42.939865,-5.709256,-5.770491,No Log,No Log,No Log


TrainOutput(global_step=21, training_loss=0.358171101127352, metrics={'train_runtime': 353.5213, 'train_samples_per_second': 1.731, 'train_steps_per_second': 0.059, 'total_flos': 0.0, 'train_loss': 0.358171101127352, 'epoch': 3.0})

In [47]:
from transformers import TextStreamer
import torch

# Pick a sample from your dataset
example = dataset_dpo[3]

# Use the pre-built prompt
prompt_text = example["prompt"]

# Convert to tensor
input_ids = tokenizer(prompt_text, return_tensors="pt").to("cuda")

# Stream the model's prediction
streamer = TextStreamer(tokenizer, skip_prompt=True)
with torch.inference_mode():
    _ = model.generate(
        **input_ids,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        top_k=20,
        streamer=streamer
    )


Explanation:
Default argument lists mutate by reference, leading to side effects.

Fixed Code:
```python
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items
```<|im_end|>


In [50]:
dataset_dpo[3]['chosen']

'Explanation:\nDefault mutable arguments can lead to unexpected behavior.\n\nFixed Code:\n```python\ndef add_item(item, items=None):\n    if items is None:\n        items = []\n    items.append(item)\n    return items\n```'